In [4]:
import gradio as gr
from ultralytics import YOLO
import cv2
import numpy as np
import tempfile
import os

# 加载模型
model = YOLO("yolo11n.pt")

def img_workflow(img, conf, iou):
    if img is None:
        return img
    results = model(img, conf=conf, iou=iou)
    result_img = results[0].plot()
    return result_img

def video_workflow(video_path, conf, iou):
    if not video_path:
        yield None, "请先上传视频"
        return

    # 打开视频文件
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        yield None, "无法打开视频文件"
        return

    # 获取视频属性
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # 创建临时输出视频文件
    target_dir = r"D:\study\yolo\newyolo\runs"
    os.makedirs(target_dir, exist_ok=True)

    # 在指定目录创建临时文件
    temp_file = tempfile.NamedTemporaryFile(suffix='.mp4', dir=target_dir, delete=False)
    out_path = temp_file.name

    # 视频写入器
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (frame_width, frame_height))

    frame_count = 0

    # 逐帧处理
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # YOLO检测
        results = model(frame, conf=conf, iou=iou)
        result_frame = results[0].plot()

        # 写入输出视频
        out.write(result_frame)

        # 转换颜色格式用于Gradio显示（BGR→RGB）
        result_frame_rgb = cv2.cvtColor(result_frame, cv2.COLOR_BGR2RGB)

        frame_count += 1
        progress = f"处理中... {frame_count}/{total_frames} 帧 ({100 * frame_count / total_frames:.1f}%)"

        # 实时更新图像和进度文本
        yield result_frame_rgb, progress

    # 释放资源
    cap.release()
    out.release()

    # 最终返回：显示最后一帧+完成提示
    final_message = f"处理完成！共 {total_frames} 帧。\n视频已保存至: {out_path}"
    yield result_frame_rgb, final_message


def camera_workflow(conf, iou):
    """生成器函数：实时摄像头检测"""
    print("正在启动摄像头... 刷新页面可停止")

    # 打开摄像头
    cap = cv2.VideoCapture(0)  # 0 表示默认摄像头
    if not cap.isOpened():
        yield np.zeros((480, 640, 3), dtype=np.uint8), "无法打开摄像头，请检查设备连接和浏览器权限"
        return

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # YOLO检测
            results = model(frame, conf=conf, iou=iou)
            result_frame = results[0].plot()

            # 转换颜色格式用于Gradio显示（BGR→RGB）
            result_frame_rgb = cv2.cvtColor(result_frame, cv2.COLOR_BGR2RGB)

            yield result_frame_rgb, "摄像头实时运行中... 刷新页面可停止"
    finally:
        # 确保释放摄像头资源
        cap.release()


with gr.Blocks() as demo:
    gr.Markdown("## YOLO11 目标检测演示")

    with gr.Tabs():
        with gr.TabItem("图片检测"):
            with gr.Row():
                inp_img = gr.Image(type="numpy", label="上传图像")
                out_img = gr.Image(type="numpy", label="输出图像")

            with gr.Row():
                conf1 = gr.Slider(0, 1, value=0.25, label="置信度阈值 (conf)")
                iou1 = gr.Slider(0, 1, value=0.45, label="IOU阈值 (iou)")
                btn1 = gr.Button("运行检测", variant="primary")

            btn1.click(img_workflow, [inp_img, conf1, iou1], out_img)

        with gr.TabItem("视频检测"):
            with gr.Row():
                inp_vid = gr.Video(label="上传视频")

            # 实时预览区域
            with gr.Row():
                out_vid_frame = gr.Image(label="实时预览", type="numpy")
                out_vid_text = gr.Textbox(label="处理状态", lines=2)

            with gr.Row():
                conf2 = gr.Slider(0, 1, value=0.25, label="置信度阈值 (conf)")
                iou2 = gr.Slider(0, 1, value=0.45, label="IOU阈值 (iou)")
                btn2 = gr.Button("运行检测", variant="primary")

            # 生成器函数绑定到多个输出
            btn2.click(
                video_workflow,
                [inp_vid, conf2, iou2],
                [out_vid_frame, out_vid_text]
            )

        with gr.TabItem("摄像头实时检测"):
            gr.Markdown("### 实时摄像头目标检测")

            # 实时预览区域
            with gr.Row():
                camera_frame = gr.Image(label="实时画面", type="numpy")
                camera_status = gr.Textbox(label="状态", lines=2)

            with gr.Row():
                conf3 = gr.Slider(0, 1, value=0.5, label="置信度阈值 (conf)")
                iou3 = gr.Slider(0, 1, value=0.45, label="IOU阈值 (iou)")

            with gr.Row():
                start_btn = gr.Button("开始检测", variant="primary")

            # 绑定事件
            start_btn.click(
                camera_workflow,
                [conf3, iou3],
                [camera_frame, camera_status]
            )

            gr.Markdown("""
            **使用说明**：
            - 点击"开始检测"启动摄像头（首次使用需要授权浏览器摄像头权限）
            - 可实时调整 conf 和 iou 参数（调整后需重新点击开始）
            - **停止方法**：刷新页面或关闭浏览器标签页
            - 如果摄像头无法打开，请检查设备连接和浏览器权限设置
            """)


demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Missing file: C:\Users\zwx\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\zwx\.cache\huggingface\gradio\frpc
